Thiès Weather Data Lake : De l'Ingestion API au Dashboard Analytique

In [5]:
import json
from datetime import datetime
import requests

# Coordonnées GPS de Thiès, Sénégal
LATITUDE = 14.7910
LONGITUDE = -16.9258

# URL de l'API Open-Meteo pour récupérer la température, l'humidité et les précipitations actuelles
url = f"https://api.open-meteo.com/v1/forecast?latitude={LATITUDE}&longitude={LONGITUDE}&current=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"

response = requests.get(url)

if response.status_code == 200:
  data = response.json()
  print("Données récupérées avec succès !")
  print(json.dumps(data, indent=4))
else:
  print(f"Erreur lors de l'appel API : {response.status_code}")

Données récupérées avec succès !
{
    "latitude": 14.797891,
    "longitude": -16.927368,
    "generationtime_ms": 0.04887580871582031,
    "utc_offset_seconds": 0,
    "timezone": "GMT",
    "timezone_abbreviation": "GMT",
    "elevation": 71.0,
    "current_units": {
        "time": "iso8601",
        "interval": "seconds",
        "temperature_2m": "\u00b0C",
        "relative_humidity_2m": "%",
        "precipitation": "mm",
        "wind_speed_10m": "km/h"
    },
    "current": {
        "time": "2026-10-19T23:30",
        "interval": 900,
        "temperature_2m": 26.4,
        "relative_humidity_2m": 91,
        "precipitation": 0.0,
        "wind_speed_10m": 2.7
    }
}


In [6]:
from datetime import datetime
import json
import os
import requests

# --- CONFIGURATION ---
LATITUDE = 14.7910
LONGITUDE = -16.9258
LOCAL_DIR = "data_meteo"  # Nom du dossier local sur ton PC

# 1. Récupération des données depuis l'API
url = f"https://api.open-meteo.com/v1/forecast?latitude={LATITUDE}&longitude={LONGITUDE}&current=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"
response = requests.get(url)

if response.status_code == 200:
  data = response.json()
  print("Données météo récupérées avec succès !")

  # 2. Préparation d'une structure de dossiers propre (Similaire à un Data Lake S3)
  now = datetime.utcnow()
  year = now.strftime("%Y")
  month = now.strftime("%m")
  day = now.strftime("%d")
  timestamp = now.strftime("%Y%m%d_%H%M%S")

  # Chemin du dossier : data_meteo/raw/year=2026/month=10/day=19/
  folder_path = os.path.join(LOCAL_DIR, "raw", f"year={year}", f"month={month}", f"day={day}")
  os.makedirs(folder_path, exist_ok=True)  # Crée les dossiers s'ils n'existent pas

  # Chemin complet du fichier JSON
  file_path = os.path.join(folder_path, f"meteo_{timestamp}.json")

  # 3. Enregistrement du fichier en local
  with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

  print(f"Fichier sauvegardé avec succès en local : {file_path}")

else:
  print(f"Erreur lors de l'appel API : {response.status_code}")

Données météo récupérées avec succès !
Fichier sauvegardé avec succès en local : data_meteo\raw\year=2026\month=10\day=19\meteo_20261019_234259.json


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12928\522891457.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


In [7]:
from datetime import datetime
import json
import os

# --- CONFIGURATION ---
LOCAL_DIR = "data_meteo"
now = datetime.utcnow()
year = now.strftime("%Y")
month = now.strftime("%m")
day = now.strftime("%d")

# 1. Retrouver le fichier brut que l'on vient de créer
raw_folder = os.path.join(LOCAL_DIR, "raw", f"year={year}", f"month={month}", f"day={day}")

# On liste les fichiers du dossier pour prendre le plus récent
fichiers = os.listdir(raw_folder)
dernier_fichier = sorted(fichiers)[-1]
chemin_brut = os.path.join(raw_folder, dernier_fichier)

print(f"Lecture du fichier brut : {chemin_brut}")

# 2. Lire le contenu du fichier JSON brut
with open(chemin_brut, "r", encoding="utf-8") as f:
  data_brute = json.load(f)

# 3. Transformer et nettoyer les données (Extraction des champs clés)
current_data = data_brute.get("current", {})
units = data_brute.get("current_units", {})

donnee_transformee = {
    "latitude": data_brute.get("latitude"),
    "longitude": data_brute.get("longitude"),
    "timestamp_mesure": current_data.get("time"),
    "temperature_celsius": current_data.get("temperature_2m"),
    "humidite_pourcentage": current_data.get("relative_humidity_2m"),
    "precipitation_mm": current_data.get("precipitation"),
    "vent_kmh": current_data.get("wind_speed_10m"),
    "date_traitement": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
}

print("Données transformées :")
print(donnee_transformee)

# 4. Sauvegarder dans un dossier 'processed' (Zone propre)
processed_folder = os.path.join(
    LOCAL_DIR, "processed", f"year={year}", f"month={month}", f"day={day}"
)
os.makedirs(processed_folder, exist_ok=True)

chemin_processed = os.path.join(processed_folder, f"clean_{dernier_fichier}")
with open(chemin_processed, "w", encoding="utf-8") as f:
  json.dump(donnee_transformee, f, ensure_ascii=False, indent=4)

print(f"Fichier nettoyé sauvegardé avec succès : {chemin_processed}")

Lecture du fichier brut : data_meteo\raw\year=2026\month=10\day=19\meteo_20261019_234259.json
Données transformées :
{'latitude': 14.797891, 'longitude': -16.927368, 'timestamp_mesure': '2026-10-19T23:30', 'temperature_celsius': 26.4, 'humidite_pourcentage': 91, 'precipitation_mm': 0.0, 'vent_kmh': 2.7, 'date_traitement': '2026-10-19 23:42:59'}
Fichier nettoyé sauvegardé avec succès : data_meteo\processed\year=2026\month=10\day=19\clean_meteo_20261019_234259.json


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12928\3728362623.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12928\3728362623.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_traitement": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),


In [8]:
import json
import os
import pandas as pd

# 1. Chemin vers ton dossier 'processed'
dossier_processed = "data_meteo/processed"

fichiers_a_lire = []
for root, dirs, files in os.walk(dossier_processed):
  for file in files:
    if file.endswith(".json"):
      fichiers_a_lire.append(os.path.join(root, file))

# 2. Charger les fichiers JSON avec le module json standard
data_list = []
for f in fichiers_a_lire:
  with open(f, "r", encoding="utf-8") as file_ouvert:
    data_list.append(json.load(file_ouvert))

# 3. Convertir la liste de dictionnaires en DataFrame Pandas
df_meteo = pd.DataFrame(data_list)

print("--- Aperçu des données chargées avec succès ---")
print(df_meteo)

# 4. Afficher les colonnes clés
print("\n--- Température et Humidité actuelles ---")
print(
    df_meteo[[
        "timestamp_mesure",
        "temperature_celsius",
        "humidite_pourcentage",
        "vent_kmh",
    ]]
)

--- Aperçu des données chargées avec succès ---
    latitude  longitude  timestamp_mesure  temperature_celsius  \
0  14.797891 -16.927368  2026-10-19T22:15                 26.8   
1  14.797891 -16.927368  2026-10-19T23:30                 26.4   

   humidite_pourcentage  precipitation_mm  vent_kmh      date_traitement  
0                    90               0.0       2.0  2026-10-19 22:30:44  
1                    91               0.0       2.7  2026-10-19 23:42:59  

--- Température et Humidité actuelles ---
   timestamp_mesure  temperature_celsius  humidite_pourcentage  vent_kmh
0  2026-10-19T22:15                 26.8                    90       2.0
1  2026-10-19T23:30                 26.4                    91       2.7


In [10]:
from datetime import datetime, timedelta
import json
import os

dossier_processed = "data_meteo/processed/year=2026/month=10/day=19"
os.makedirs(dossier_processed, exist_ok=True)

# Simuler quelques heures de météo à Thiès
historique_fictif = [
    {
        "latitude": 14.797891,
        "longitude": -16.927368,
        "timestamp_mesure": "2026-10-19T18:00",
        "temperature_celsius": 29.5,
        "humidite_pourcentage": 75,
        "precipitation_mm": 0.0,
        "vent_kmh": 4.0,
        "date_traitement": "2026-10-19 18:05:00",
    },
    {
        "latitude": 14.797891,
        "longitude": -16.927368,
        "timestamp_mesure": "2026-10-19T19:00",
        "temperature_celsius": 28.2,
        "humidite_pourcentage": 80,
        "precipitation_mm": 0.0,
        "vent_kmh": 3.5,
        "date_traitement": "2026-10-19 19:05:00",
    },
    {
        "latitude": 14.797891,
        "longitude": -16.927368,
        "timestamp_mesure": "2026-10-19T20:00",
        "temperature_celsius": 27.5,
        "humidite_pourcentage": 85,
        "precipitation_mm": 0.0,
        "vent_kmh": 3.0,
        "date_traitement": "2026-10-19 20:05:00",
    },
    {
        "latitude": 14.797891,
        "longitude": -16.927368,
        "timestamp_mesure": "2026-10-19T21:00",
        "temperature_celsius": 27.0,
        "humidite_pourcentage": 88,
        "precipitation_mm": 0.0,
        "vent_kmh": 2.5,
        "date_traitement": "2026-10-19 21:05:00",
    },
    {
        "latitude": 14.797891,
        "longitude": -16.927368,
        "timestamp_mesure": "2026-10-19T22:15",
        "temperature_celsius": 26.8,
        "humidite_pourcentage": 90,
        "precipitation_mm": 0.0,
        "vent_kmh": 2.0,
        "date_traitement": "2026-10-19 22:30:44",
    },
]

# Sauvegarde des fichiers JSON de test
for i, point in enumerate(historique_fictif):
  chemin = os.path.join(dossier_processed, f"historique_test_{i}.json")
  with open(chemin, "w", encoding="utf-8") as f:
    json.dump(point, f, ensure_ascii=False, indent=4)

print("Historique de test généré avec succès !")

Historique de test généré avec succès !


In [ ]:
from datetime import datetime
import json
import os
import time
import requests

LOCAL_DIR = "data_meteo"
url = "https://api.open-meteo.com/v1/forecast?latitude=14.7979&longitude=-16.9274&current=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"

print(
    "🔄 Démarrage du pipeline automatisé (appuie sur Ctrl+C pour arrêter)..."
)

while True:
  try:
    # 1. Ingestion (Raw)
    response = requests.get(url)
    data_brute = response.json()

    now = datetime.utcnow()
    year, month, day = (
        now.strftime("%Y"),
        now.strftime("%m"),
        now.strftime("%d"),
    )

    raw_folder = os.path.join(
        LOCAL_DIR, "raw", f"year={year}", f"month={month}", f"day={day}"
    )
    os.makedirs(raw_folder, exist_ok=True)

    timestamp_str = now.strftime("%Y%m%d_%H%M%S")
    chemin_brut = os.path.join(raw_folder, f"meteo_{timestamp_str}.json")

    with open(chemin_brut, "w", encoding="utf-8") as f:
      json.dump(data_brute, f, ensure_ascii=False, indent=4)

    # 2. Transformation (Processed)
    current_data = data_brute.get("current", {})
    donnee_transformee = {
        "latitude": data_brute.get("latitude"),
        "longitude": data_brute.get("longitude"),
        "timestamp_mesure": current_data.get("time"),
        "temperature_celsius": current_data.get("temperature_2m"),
        "humidite_pourcentage": current_data.get("relative_humidity_2m"),
        "precipitation_mm": current_data.get("precipitation"),
        "vent_kmh": current_data.get("wind_speed_10m"),
        "date_traitement": now.strftime("%Y-%m-%d %H:%M:%S"),
    }

    processed_folder = os.path.join(
        LOCAL_DIR, "processed", f"year={year}", f"month={month}", f"day={day}"
    )
    os.makedirs(processed_folder, exist_ok=True)

    chemin_processed = os.path.join(
        processed_folder, f"clean_meteo_{timestamp_str}.json"
    )
    with open(chemin_processed, "w", encoding="utf-8") as f:
      json.dump(donnee_transformee, f, ensure_ascii=False, indent=4)

    print(
        f"[{now.strftime('%H:%M:%S'
        )}] Données récupérées et nettoyées avec succès !"
    )

  except Exception as e:
    print(f"Erreur lors de la récupération : {e}")

  # Attendre 1 heure (3600 secondes) avant la prochaine exécution
  # (Tu peux remplacer par 60 secondes pour tester rapidement)
  time.sleep(3600)